In [ ]:
import sys

sys.path.append("/home/ubuntu/work/therapeutic_accelerator/custom_packages/utils")
sys.path.append("/home/ubuntu/work/therapeutic_accelerator/custom_packages/database")

In [ ]:
import pandas as pd
import sqlalchemy as sa
import ast
from glob import glob
import numpy as np
from tqdm import tqdm
import warnings
import logging

warnings.filterwarnings('ignore')

max_sequence_length = 1200
embedding_size = 200

In [ ]:
import chromaDB as cbd # specter_ef, create_text_splitter, create_chroma_client
from db_tools import db_connection
from utils import import_config
from transformers import AutoModel, AutoTokenizer

config, keys = import_config()

# Chroma setup --------------------------------------------------------------------------------
chroma_server_host = "3.88.178.230"

chroma_client = cbd.create_chroma_client(chroma_server_host)

# Modelish stuff --------------------------------------------------------------------------------

model = AutoModel.from_pretrained("allenai/specter")
tokenizer = AutoTokenizer.from_pretrained("allenai/specter")

# Create embeddings function with specter model
specter_embeder = cbd.specter_ef(model, tokenizer)

specter_embeder.create_text_splitter()


In [ ]:
# Working Collection
collection = chroma_client.get_or_create_collection("fulltext_specter", embedding_function=specter_embeder)

print(collection.count())

In [ ]:
# from IPython.utils import io

# with io.capture_output() as captured:

In [ ]:
# # Create dask cluster
# dask.config.set(scheduler='processes')  # overwrite default with multiprocessing scheduler

# cluster = distributed.LocalCluster(name='local', n_workers=7, memory_limit = '4GiB', threads_per_worker=2)  # Launches a scheduler and workers locally

# client = distributed.client._get_global_client() or distributed.Client(cluster)

# client

## Now with Dask

Functions to clean up dataframes

In [ ]:
def split_text(df):
    """ Split text into chunks """
    df = df.assign(split_text = df['text'].apply(text_splitter.split_text))
    df = df.drop(columns = 'text')
    return df

# ------------------------------------------------------------------------------------
def create_doc(split_text, corpusid):
    """ Create documents for each chunk """
        
    try:
        docs = {
            "documents": split_text, # list of all documents [doc1, doc2, doc3, ...]
            'ids': [f'{corpusid}-{i}' for i in range(len(split_text))], # list of all ids [id1, id2, id3, ...]
            'metadatas': [{'corpusid': int(corpusid), 'chunk': i} for i in range(len(split_text))] # list of dictionaries with metadata for each document
        }
        return docs

    except Exception as e:
        logging.error(e)
        
def df_create_doc(df):
    """ Used for mapping partitions
    
    Takes dataframe
    
    Returns a series
    
    """
    return df.apply(lambda x: create_doc(x['split_text'], x['corpusid']), axis=1)

def mp_create_doc(ddf): 
    """ Used for mapping partitions"""
    return ddf.apply(df_create_doc, axis = 1)

# ------------------------------------------------------------------------------------
# Add documents to collection
def add_to_collection(docs, collection):
    """ Add documents to collection """
    try:
        collection.add(**docs)
    except Exception as e:
        logging.error(e)
        
def ddf_add_to_collection(series, **kwargs):
    """ Add documents to collection """
    return series.apply(add_to_collection)

In [ ]:
# # Read in fulltext from csvs for dask
# ft = dd.read_parquet('/home/ubuntu/work/data/fulltext_parquets/fulltext-*.parquet', sample=10000000,
#                      sample_rows=10,
#                      lineterminator=None,
#                      dtype={'corpusid': 'int', 'text': 'object'})

# # Cleanup dataframes
# ft = ft.map_partitions(pd.DataFrame.dropna, subset='text')

# ft = ft.map_partitions(pd.DataFrame.drop_duplicates, subset='text')

# ft = ft.map_partitions(pd.DataFrame.reset_index, drop=True)

# ft = ft.persist()


In [ ]:
# creates futures to then act on to create tree of dependencies
# results = ft.compute(optimize_graph = True, scheduler='processes', num_workers=7)

In [ ]:
# futures_split_text = [client.submit(split_text, f) for f in results]

In [ ]:
# docs = [client.submit(df_create_doc, f) for f in futures_split_text]

In [ ]:
# import os

# # get number of files in a folder
# def get_num_files(path):
#     """ Get number of files in a folder """
#     return len([name for name in os.listdir(path) if os.path.isfile(os.path.join(path, name))])

# get_num_files('/home/ubuntu/work/data/fulltext_docs_csvs')


In [ ]:
# def create_csv(d):
#     """ Create csvs for each chunk """
#     df = pd.DataFrame(d)
#     i = get_num_files('/home/ubuntu/work/data/fulltext_docs_csvs') + 1
    
#     df.to_csv(f'/home/ubuntu/work/data/fulltext_docs_csvs/fulltext_doc_{i}.csv', index=False)
    
#     del d

In [ ]:
# Write out documents for easier uploading later
# csvs  = [client.submit(create_csv, d) for d in docs]

# Construct Chroma Collection

In [ ]:
def remove_exisiting(df):
    """ Remove existing ids from dataframe """
    # get unique ids from collection that are the corpus ids
    existing_ids = np.unique(pd.DataFrame(collection.get(include = ['metadatas']))['ids'].str.split('-',expand=True).rename(columns={0:'id',1:'part'})['id'].tolist()).tolist()
    print('N existing ids: ',len(existing_ids))
    print(df.shape)
    
    df = df[~df['corpusid'].astype(str).isin(existing_ids)]
    
    print(df.shape)
    
    return df

In [ ]:
existing_ids = np.unique(pd.DataFrame(collection.get(include = ['metadatas']))['ids'].str.split('-',expand=True).rename(columns={0:'id',1:'part'})['id'].tolist()).tolist()

In [ ]:
existing_ids

In [ ]:
file_list = glob('/home/ubuntu/work/data/fulltext_docs_csvs_cleaned/fulltext_doc_*.csv')

In [ ]:
# ------------------------------------------------------------------------------------
# Add documents to collection
def add_to_collection(docs):
    """ Add documents to collection """
    
    docs = ast.literal_eval(docs) # since dictionaries were stored as string
    
    # Check if corpusid already exists in collection
    checker = pd.DataFrame(ast.literal_eval(df[0])['metadatas'])['corpusid'].unique().astype(str) in existing_ids
    if checker: 
        return True
    else: 
        docs['embeddings'] = specter_embeder.embed_documents(docs['documents'])

        try:
            collection.add(**docs)
            
        except Exception as e:
            logging.error(e)

In [ ]:
remove_exisiting(df)

In [ ]:
test = 

In [ ]:
test

: 

In [ ]:
for f in tqdm(file_list): 
    # print(f)
    df = pd.read_csv(f).iloc[:, 0]
    
    df.apply(add_to_collection)

In [ ]:
collection.count()